## Advanced Model: Alternating Least Squares (ALS) for Implicit Feedback

In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
import random

c:\Users\jerem\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load data
train_df = pd.read_csv('../data/train.csv')
test_users_df = pd.read_csv('../data/data_target_users_test.csv')

In [4]:
# Prepare user-item matrix (sparse)
# Map user_ids and item_ids to indices
user_ids = train_df['user_id'].unique()
item_ids = train_df['item_id'].unique()
user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}
idx_to_item_id = {idx: iid for iid, idx in item_id_to_idx.items()}

In [ ]:
# Create sparse matrix (rows: users, cols: items, values: 1 for interactions)
rows = [user_id_to_idx[uid] for uid in train_df['user_id']]
cols = [item_id_to_idx[iid] for iid in train_df['item_id']]
data = [1] * len(train_df)
user_item_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(item_ids)))


In [7]:
# Train ALS model (tune parameters: factors=100, regularization=0.01, iterations=20, alpha=40 for implicit weighting)
model = AlternatingLeastSquares(factors=100, regularization=0.01, iterations=20, alpha=40)
model.fit(user_item_matrix)

100%|██████████| 20/20 [00:04<00:00,  4.84it/s]


In [8]:
# Function to get ALS recommendations for a user
def get_als_recommendations(user_id, model, user_item_matrix, user_id_to_idx, idx_to_item_id, user_interactions, top_items):
    if user_id not in user_id_to_idx:
        # Cold-start: Fall back to popularity
        interacted_items = set(user_interactions.get(user_id, []))
        recommendations = [item for item in top_items.index if item not in interacted_items][:10]
        return ' '.join(recommendations)
    
    user_idx = user_id_to_idx[user_id]
    # Get recommendations (exclude already interacted items)
    recs_indices, _ = model.recommend(user_idx, user_item_matrix[user_idx], N=10)
    recommendations = [idx_to_item_id[idx] for idx in recs_indices]
    return ' '.join(recommendations)

In [9]:
# Prepare user interactions dict
user_interactions = train_df.groupby('user_id')['item_id'].apply(list).to_dict()

# Get top 10 global items for cold-start fallback
item_popularity = train_df['item_id'].value_counts().sort_values(ascending=False)
top_10_items = item_popularity.head(10)

top_10_items

item_id
0316666343    427
0385504209    330
0312195516    241
0142001740    214
059035342X    206
0060928336    204
0446672211    197
0345337662    183
0452282152    173
0316601950    169
Name: count, dtype: int64

In [10]:
# Generate recommendations for each test user
recommendations = []
for user_id in test_users_df['user_id']:
    recs = get_als_recommendations(user_id, model, user_item_matrix, user_id_to_idx, idx_to_item_id, user_interactions, top_10_items)
    recommendations.append({'user_id': user_id, 'item_id': recs})

In [11]:
# Create submission DataFrame
submission_df = pd.DataFrame(recommendations)
submission_df.to_csv('../data/als_submission.csv', index=False)